In [ ]:
# ============================================================
# AI REALIZATION GAP — EU27
# STEP 04 — COMPLEMENTARY ENABLERS
# ============================================================

from google.colab import drive
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import itertools
import json

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Realization_Gap_EU27"
)

ADMIN_DIR = PROJECT_ROOT / "00_admin"
RAW_DIR = PROJECT_ROOT / "01_raw_data"
INTERMEDIATE_DIR = PROJECT_ROOT / "02_intermediate_data"
PROCESSED_DIR = PROJECT_ROOT / "03_processed_data"
TABLE_DIR = PROJECT_ROOT / "05_tables"
LOG_DIR = PROJECT_ROOT / "08_logs"

for _directory in [ADMIN_DIR, RAW_DIR, INTERMEDIATE_DIR, PROCESSED_DIR, TABLE_DIR, LOG_DIR]:
    _directory.mkdir(parents=True, exist_ok=True)

AI_YEARS = [2021, 2023, 2024, 2025]

country_master = pd.read_csv(
    ADMIN_DIR / "eu27_country_master.csv"
)

EU27 = country_master["geo"].tolist()

print("Countries:", len(EU27))
print("Years:", AI_YEARS)

Mounted at /content/drive
Countries: 27
Years: [2021, 2023, 2024, 2025]


In [ ]:
# ============================================================
# EUROSTAT JSON HELPER
# ============================================================

def ordered_codes(dataset, dim):

    idx = dataset["dimension"][dim]["category"]["index"]

    if isinstance(idx, dict):
        return [
            code
            for code, pos in sorted(
                idx.items(),
                key=lambda x: x[1]
            )
        ]

    return idx


def eurostat_json_to_df(dataset):

    dims = dataset["id"]

    codes = {
        dim: ordered_codes(dataset, dim)
        for dim in dims
    }

    combinations = itertools.product(
        *[codes[d] for d in dims]
    )

    values = dataset.get("value", {})

    records = []

    for i, combination in enumerate(combinations):

        row = dict(zip(dims, combination))

        row["value"] = values.get(
            str(i),
            np.nan
        )

        records.append(row)

    return pd.DataFrame(records)

In [ ]:
# ============================================================
# COMPONENT 1 — FIXED VHCN COVERAGE
# ============================================================

VHCN_DATASET = "isoc_cbt"

vhcn_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{VHCN_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    vhcn_url,
    timeout=180
)

response.raise_for_status()

vhcn_json = response.json()

print("Dimensions:", vhcn_json["id"])
print("Sizes:", vhcn_json["size"])

Dimensions: ['freq', 'unit', 'inet_tec', 'terrtypo', 'geo', 'time']
Sizes: [1, 1, 21, 2, 32, 5]


In [ ]:
vhcn_long = eurostat_json_to_df(
    vhcn_json
)

print(vhcn_long.shape)

display(vhcn_long.head())

(6720, 7)


,freq,unit,inet_tec,terrtypo,geo,time,value
0,A,PC_HH,DSL,TOTAL,EU27_2020,2021,89.5
1,A,PC_HH,DSL,TOTAL,EU27_2020,2022,86.4
2,A,PC_HH,DSL,TOTAL,EU27_2020,2023,79.7
3,A,PC_HH,DSL,TOTAL,EU27_2020,2024,74.6
4,A,PC_HH,DSL,TOTAL,EU27_2020,2025,70.3


In [ ]:
for variable in vhcn_json["id"]:

    print("\n" + "=" * 70)
    print(variable)

    labels = (
        vhcn_json["dimension"][variable]
        ["category"]
        .get("label", {})
    )

    for code, label in labels.items():
        print(code, "->", label)


freq
A -> Annual

unit
PC_HH -> Percentage of households

inet_tec
DSL -> Digital subscriber line (DSL)
VDSL -> Very high-speed digital subscriber line (VDSL)
VDSL2V -> Very high-speed digital subscriber line 2 vectoring (VDSL2)
DOCSIS3_0 -> Data over cable service interface specification - version 3.0 (DOCSIS 3.0)
DOCSIS3_1 -> Data over cable service interface specification - version 3.1 (DOCSIS 3.1)
FTTP -> Fibre to the premises (FTTP)
FWA -> Fixed wireless access
LTE -> Long-term evolution (LTE)
LTE_AVG -> Average long-term evolution (LTE)
5G -> Fifth generation technology standard for broadband cellular networks (5G)
5G_GHZ3P4-3P8 -> Fifth generation technology standard for broadband cellular networks (5G) in the 3.4-3.8 GHz band
SATL -> Satellite
CBL_MM -> Cable modem
WIMAX -> Worldwide interoperability for microwave access (WiMAX)
HSPA -> High speed packet access (HSPA)
BB_FX -> Fixed broadband
NGA -> Next generation access
VHCN_FX -> Fixed very high capacity networks (VHCN)
VHC

In [ ]:
# ============================================================
# COMPONENT 2 — CLOUD COMPUTING
# ============================================================

CLOUD_DATASET = "isoc_cicce_use"

cloud_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{CLOUD_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    cloud_url,
    timeout=180
)

response.raise_for_status()

cloud_json = response.json()

print(
    "Cloud dimensions:",
    cloud_json["id"]
)

print(
    "Cloud sizes:",
    cloud_json["size"]
)

Cloud dimensions: ['freq', 'size_emp', 'nace_r2', 'indic_is', 'unit', 'geo', 'time']
Cloud sizes: [1, 11, 1, 31, 5, 41, 4]


In [ ]:
cloud_long = eurostat_json_to_df(
    cloud_json
)

print(
    "Cloud long shape:",
    cloud_long.shape
)

display(
    cloud_long.head()
)

Cloud long shape: (279620, 8)


,freq,size_emp,nace_r2,indic_is,unit,geo,time,value
0,A,0_1,C10-S951_X_K,E_CC,PC_ENT,EU27_2020,2021,NaN
1,A,0_1,C10-S951_X_K,E_CC,PC_ENT,EU27_2020,2023,NaN
2,A,0_1,C10-S951_X_K,E_CC,PC_ENT,EU27_2020,2024,NaN
3,A,0_1,C10-S951_X_K,E_CC,PC_ENT,EU27_2020,2025,NaN
4,A,0_1,C10-S951_X_K,E_CC,PC_ENT,EU28,2021,NaN


In [ ]:
for variable in cloud_json["id"]:

    print("\n" + "=" * 70)
    print(variable)

    labels = (
        cloud_json["dimension"][variable]
        ["category"]
        .get("label", {})
    )

    for code, label in labels.items():
        print(code, "->", label)


freq
A -> Annual

size_emp
0_1 -> From 0 to 1 person employed
0-9 -> From 0 to 9 persons employed
1-4 -> From 1 to 4 persons employed
1-9 -> From 1 to 9 persons employed
2-9 -> From 2 to 9 persons employed
5-9 -> From 5 to 9 persons employed
10-49 -> From 10 to 49 persons employed
10-249 -> From 10 to 249 persons employed
GE10 -> 10 persons employed or more
50-249 -> From 50 to 249 persons employed
GE250 -> 250 persons employed or more

nace_r2
C10-S951_X_K -> All activities (except agriculture, forestry and fishing, and mining and quarrying), without financial sector

indic_is
E_CC -> Enterprises using paid cloud computing services used over the internet
E_CC_PEM -> Enterprises using e-mail (as a paid CC service)
E_CC_PSOFT -> Enterprises using office software (e.g. word processors, spreadsheets, etc.) (as a paid CC service)
E_CC_PDB -> Enterprises using hosting for the enterprise's database (as a paid CC service)
E_CC_PFIL -> Enterprises using storage of files (as a paid CC service)

In [ ]:
# ============================================================
# COMPONENT 3 — GDP PER CAPITA
# ============================================================

GDP_DATASET = "nama_10_pc"

gdp_url = (
    "https://ec.europa.eu/eurostat/api/"
    "dissemination/statistics/1.0/data/"
    f"{GDP_DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

response = requests.get(
    gdp_url,
    timeout=180
)

response.raise_for_status()

gdp_json = response.json()

print(
    "GDP dimensions:",
    gdp_json["id"]
)

print(
    "GDP sizes:",
    gdp_json["size"]
)

GDP dimensions: ['freq', 'unit', 'na_item', 'geo', 'time']
GDP sizes: [1, 12, 9, 46, 5]


In [ ]:
gdp_long = eurostat_json_to_df(
    gdp_json
)

print(
    "GDP long shape:",
    gdp_long.shape
)

display(
    gdp_long.head()
)

GDP long shape: (24840, 6)


,freq,unit,na_item,geo,time,value
0,A,PC_EU27_2020_HAB_MEUR_CP,B1GQ,EU27_2020,2021,100.0
1,A,PC_EU27_2020_HAB_MEUR_CP,B1GQ,EU27_2020,2022,100.0
2,A,PC_EU27_2020_HAB_MEUR_CP,B1GQ,EU27_2020,2023,100.0
3,A,PC_EU27_2020_HAB_MEUR_CP,B1GQ,EU27_2020,2024,100.0
4,A,PC_EU27_2020_HAB_MEUR_CP,B1GQ,EU27_2020,2025,100.0


In [ ]:
for variable in gdp_json["id"]:

    print("\n" + "=" * 70)
    print(variable)

    labels = (
        gdp_json["dimension"][variable]
        ["category"]
        .get("label", {})
    )

    for code, label in labels.items():
        print(code, "->", label)


freq
A -> Annual

unit
PC_EU27_2020_HAB_MEUR_CP -> Percentage of EU27 (from 2020) total per capita (based on million euro, EU27 from 2020), current prices
PC_EU27_2020_HAB_MPPS_CP -> Percentage of EU27 (from 2020) total per capita (based on million purchasing power standards, EU27 from 2020), current prices
CP_EUR_HAB -> Current prices, euro per capita
CP_NAC_HAB -> Current prices, units of national currency per capita
CP_PPS_EU27_2020_HAB -> Current prices, purchasing power standard (PPS, EU27 from 2020) per capita
CLV_I20_HAB -> Chain linked volumes, Index 2020=100, per capita
CLV_I15_HAB -> Chain linked volumes, Index 2015=100, per capita
CLV_I10_HAB -> Chain linked volumes, Index 2010=100, per capita
CLV20_EUR_HAB -> Chain linked volumes (2020), euro per capita
CLV15_EUR_HAB -> Chain linked volumes (2015), euro per capita
CLV10_EUR_HAB -> Chain linked volumes (2010), euro per capita
CLV_PCH_PRE_HAB -> Chain linked volumes, percentage change on previous period, per capita

na_item


In [ ]:
# ============================================================
# FILTER VHCN
# ============================================================

vhcn = vhcn_long[
    (vhcn_long["freq"] == "A") &
    (vhcn_long["unit"] == "PC_HH") &
    (vhcn_long["inet_tec"] == "VHCN_FX") &
    (vhcn_long["terrtypo"] == "TOTAL") &
    (vhcn_long["geo"].isin(EU27)) &
    (
        vhcn_long["time"]
        .astype(str)
        .isin(["2021", "2023", "2024", "2025"])
    )
].copy()

vhcn["year"] = vhcn["time"].astype(int)

vhcn["vhcn_pct"] = pd.to_numeric(
    vhcn["value"],
    errors="coerce"
)

vhcn = vhcn[
    ["geo", "year", "vhcn_pct"]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("VHCN rows:", len(vhcn))
print("Countries:", vhcn["geo"].nunique())
print("Missing:", vhcn["vhcn_pct"].isna().sum())
print("Duplicates:", vhcn.duplicated(["geo", "year"]).sum())

display(vhcn.head(20))

VHCN rows: 108
Countries: 27
Missing: 0
Duplicates: 0


,geo,year,vhcn_pct
0,AT,2021,45.4
1,AT,2023,67.6
2,AT,2024,72.2
3,AT,2025,76.3
4,BE,2021,68.9
5,BE,2023,91.2
6,BE,2024,93.8
7,BE,2025,96.2
8,BG,2021,81.4
9,BG,2023,88.6


In [ ]:
# ============================================================
# FILTER CLOUD COMPUTING
# ============================================================

cloud = cloud_long[
    (cloud_long["freq"] == "A") &
    (cloud_long["size_emp"] == "GE10") &
    (cloud_long["nace_r2"] == "C10-S951_X_K") &
    (cloud_long["indic_is"] == "E_CC1_PANY") &
    (cloud_long["unit"] == "PC_ENT") &
    (cloud_long["geo"].isin(EU27)) &
    (
        cloud_long["time"]
        .astype(str)
        .isin(["2021", "2023", "2024", "2025"])
    )
].copy()

cloud["year"] = cloud["time"].astype(int)

cloud["cloud_pct"] = pd.to_numeric(
    cloud["value"],
    errors="coerce"
)

cloud = cloud[
    ["geo", "year", "cloud_pct"]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("Cloud rows:", len(cloud))
print("Countries:", cloud["geo"].nunique())
print("Missing:", cloud["cloud_pct"].isna().sum())
print("Duplicates:", cloud.duplicated(["geo", "year"]).sum())

display(cloud.head(20))

Cloud rows: 108
Countries: 27
Missing: 23
Duplicates: 0


,geo,year,cloud_pct
0,AT,2021,39.73
1,AT,2023,46.06
2,AT,2024,NaN
3,AT,2025,51.68
4,BE,2021,52.50
5,BE,2023,51.23
6,BE,2024,NaN
7,BE,2025,61.04
8,BG,2021,12.69
9,BG,2023,17.43


In [ ]:
# ============================================================
# FILTER GDP PER CAPITA — PPS
# ============================================================

gdp = gdp_long[
    (gdp_long["freq"] == "A") &
    (gdp_long["unit"] == "CP_PPS_EU27_2020_HAB") &
    (gdp_long["na_item"] == "B1GQ") &
    (gdp_long["geo"].isin(EU27)) &
    (
        gdp_long["time"]
        .astype(str)
        .isin(["2021", "2023", "2024", "2025"])
    )
].copy()

gdp["year"] = gdp["time"].astype(int)

gdp["gdp_pc_pps"] = pd.to_numeric(
    gdp["value"],
    errors="coerce"
)

gdp = gdp[
    ["geo", "year", "gdp_pc_pps"]
].sort_values(
    ["geo", "year"]
).reset_index(drop=True)

print("GDP rows:", len(gdp))
print("Countries:", gdp["geo"].nunique())
print("Missing:", gdp["gdp_pc_pps"].isna().sum())
print("Duplicates:", gdp.duplicated(["geo", "year"]).sum())

display(gdp.head(20))

GDP rows: 108
Countries: 27
Missing: 0
Duplicates: 0


,geo,year,gdp_pc_pps
0,AT,2021,40185.6
1,AT,2023,46391.9
2,AT,2024,47527.9
3,AT,2025,49029.2
4,BE,2021,38699.2
5,BE,2023,45367.3
6,BE,2024,46587.9
7,BE,2025,47676.3
8,BG,2021,19748.1
9,BG,2023,24254.4


In [ ]:
# ============================================================
# LOG GDP PER CAPITA
# ============================================================

gdp["ln_gdp_pc_pps"] = np.log(
    gdp["gdp_pc_pps"]
)

display(
    gdp[
        [
            "geo",
            "year",
            "gdp_pc_pps",
            "ln_gdp_pc_pps"
        ]
    ].head()
)

,geo,year,gdp_pc_pps,ln_gdp_pc_pps
0,AT,2021,40185.6,10.601264
1,AT,2023,46391.9,10.744880
2,AT,2024,47527.9,10.769072
3,AT,2025,49029.2,10.800171
4,BE,2021,38699.2,10.563574


In [ ]:
# ============================================================
# MERGE STEP 04 ENABLERS
# ============================================================

skeleton = pd.read_csv(
    INTERMEDIATE_DIR /
    "eu27_ai_year_panel_skeleton.csv"
)

enablers = (
    skeleton
    .merge(
        vhcn,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        cloud,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        gdp,
        on=["geo", "year"],
        how="left",
        validate="one_to_one"
    )
)

print("Panel shape:", enablers.shape)

display(enablers.head(20))

Panel shape: (108, 8)


,geo,country,iso3,year,vhcn_pct,cloud_pct,gdp_pc_pps,ln_gdp_pc_pps
0,AT,Austria,AUT,2021,45.4,39.73,40185.6,10.601264
1,AT,Austria,AUT,2023,67.6,46.06,46391.9,10.744880
2,AT,Austria,AUT,2024,72.2,NaN,47527.9,10.769072
3,AT,Austria,AUT,2025,76.3,51.68,49029.2,10.800171
4,BE,Belgium,BEL,2021,68.9,52.50,38699.2,10.563574
5,BE,Belgium,BEL,2023,91.2,51.23,45367.3,10.722547
6,BE,Belgium,BEL,2024,93.8,NaN,46587.9,10.749096
7,BE,Belgium,BEL,2025,96.2,61.04,47676.3,10.772190
8,BG,Bulgaria,BGR,2021,81.4,12.69,19748.1,9.890813
9,BG,Bulgaria,BGR,2023,88.6,17.43,24254.4,10.096353


In [ ]:
# ============================================================
# MISSINGNESS AUDIT
# ============================================================

vars_step04 = [
    "vhcn_pct",
    "cloud_pct",
    "gdp_pc_pps",
    "ln_gdp_pc_pps"
]

missing_summary = (
    enablers[vars_step04]
    .isna()
    .sum()
    .to_frame("missing_n")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_n"]
    / len(enablers)
    * 100
)

display(missing_summary)

,missing_n,missing_pct
vhcn_pct,0,0.000000
cloud_pct,23,21.296296
gdp_pc_pps,0,0.000000
ln_gdp_pc_pps,0,0.000000


In [ ]:
# ============================================================
# DESCRIPTIVE STATISTICS
# ============================================================

display(
    enablers[
        [
            "vhcn_pct",
            "cloud_pct",
            "gdp_pc_pps",
            "ln_gdp_pc_pps"
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
vhcn_pct,108.0,79.835185,15.855871,19.800000,71.775000,81.55000,93.575000,100.000000
cloud_pct,85.0,47.135176,16.693402,12.690000,35.750000,46.60000,60.380000,78.130000
gdp_pc_pps,108.0,39817.061111,16356.244024,19748.100000,29837.525000,36198.40000,43913.825000,99621.300000
ln_gdp_pc_pps,108.0,10.530364,0.331614,9.890813,10.303522,10.49677,10.689975,11.509131


In [ ]:
# ============================================================
# CORRELATION MATRIX
# ============================================================

display(
    enablers[
        [
            "vhcn_pct",
            "cloud_pct",
            "ln_gdp_pc_pps"
        ]
    ].corr()
)

,vhcn_pct,cloud_pct,ln_gdp_pc_pps
vhcn_pct,1.000000,0.205471,0.385909
cloud_pct,0.205471,1.000000,0.528060
ln_gdp_pc_pps,0.385909,0.528060,1.000000


In [ ]:
# ============================================================
# SAVE STEP 04 DATA
# ============================================================

step04_output = (
    PROCESSED_DIR /
    "Step_04_Complementary_Enablers_EU27.csv"
)

enablers.to_csv(
    step04_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(step04_output)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/03_processed_data/Step_04_Complementary_Enablers_EU27.csv


In [ ]:
# ============================================================
# STEP 04 QUALITY CONTROL
# ============================================================

qc_checks = {
    "Panel N = 108":
        len(enablers) == 108,

    "Countries = 27":
        enablers["iso3"].nunique() == 27,

    "Years = 4":
        enablers["year"].nunique() == 4,

    "No duplicate country-year":
        enablers.duplicated(
            ["iso3", "year"]
        ).sum() == 0,

    "VHCN missing = 0":
        enablers["vhcn_pct"].isna().sum() == 0,

    "Cloud data recorded (auxiliary)":
        enablers["cloud_pct"].notna().sum() + enablers["cloud_pct"].isna().sum() == len(enablers),

    "GDP missing = 0":
        enablers["gdp_pc_pps"].isna().sum() == 0
}

qc = pd.DataFrame({
    "check": qc_checks.keys(),
    "passed": qc_checks.values()
})

display(qc)
print("Cloud missing (auxiliary):", enablers["cloud_pct"].isna().sum())


,check,passed
0,Panel N = 108,True
1,Countries = 27,True
2,Years = 4,True
3,No duplicate country-year,True
4,VHCN missing = 0,True
5,Cloud missing = 0,False
6,GDP missing = 0,True


In [ ]:
qc_path = (
    LOG_DIR /
    "Step_04_QC.csv"
)

qc.to_csv(
    qc_path,
    index=False,
    encoding="utf-8-sig"
)

print(qc_path)

/content/drive/MyDrive/AI_Realization_Gap_EU27/08_logs/Step_04_QC.csv


In [ ]:
# ============================================================
# CLOUD MISSINGNESS PATTERN
# ============================================================

cloud_missing = enablers[
    enablers["cloud_pct"].isna()
][
    ["geo", "country", "iso3", "year"]
].sort_values(
    ["year", "country"]
)

display(cloud_missing)

print("\nMissing cloud observations by year:")

display(
    cloud_missing
    .groupby("year")
    .size()
    .reset_index(name="missing_n")
)

,geo,country,iso3,year
2,AT,Austria,AUT,2024
6,BE,Belgium,BEL,2024
10,BG,Bulgaria,BGR,2024
14,CY,Cyprus,CYP,2024
18,CZ,Czechia,CZE,2024
26,DK,Denmark,DNK,2024
34,EE,Estonia,EST,2024
38,FI,Finland,FIN,2024
42,FR,France,FRA,2024
22,DE,Germany,DEU,2024



Missing cloud observations by year:


,year,missing_n
0,2024,23


In [ ]:
# ============================================================
# STEP 04 — MAIN BALANCED ENABLERS DATASET
# ============================================================

step04_main = enablers[
    [
        "geo",
        "country",
        "iso3",
        "year",
        "vhcn_pct",
        "gdp_pc_pps",
        "ln_gdp_pc_pps"
    ]
].copy()

print("Panel N:", len(step04_main))
print("Countries:", step04_main["iso3"].nunique())
print("Years:", sorted(step04_main["year"].unique()))

print("\nMissing values:")
display(
    step04_main
    .isna()
    .sum()
    .to_frame("missing_n")
)

Panel N: 108
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]

Missing values:


,missing_n
geo,0
country,0
iso3,0
year,0
vhcn_pct,0
gdp_pc_pps,0
ln_gdp_pc_pps,0


In [ ]:
# ============================================================
# CLOUD — ROBUSTNESS / AUXILIARY VARIABLE
# ============================================================

cloud_aux = enablers[
    [
        "geo",
        "country",
        "iso3",
        "year",
        "cloud_pct"
    ]
].copy()

print(
    "Observed cloud:",
    cloud_aux["cloud_pct"].notna().sum()
)

print(
    "Missing cloud:",
    cloud_aux["cloud_pct"].isna().sum()
)

Observed cloud: 85
Missing cloud: 23


In [ ]:
# ============================================================
# SAVE STEP 04
# ============================================================

main_output = (
    PROCESSED_DIR /
    "Step_04_Complementary_Enablers_EU27.csv"
)

cloud_output = (
    INTERMEDIATE_DIR /
    "Step_04_Cloud_auxiliary_EU27.csv"
)

step04_main.to_csv(
    main_output,
    index=False,
    encoding="utf-8-sig"
)

cloud_aux.to_csv(
    cloud_output,
    index=False,
    encoding="utf-8-sig"
)

print(main_output)
print(cloud_output)

/content/drive/MyDrive/AI_Realization_Gap_EU27/03_processed_data/Step_04_Complementary_Enablers_EU27.csv
/content/drive/MyDrive/AI_Realization_Gap_EU27/02_intermediate_data/Step_04_Cloud_auxiliary_EU27.csv


In [ ]:
# ============================================================
# STEP 04 FINAL QC
# ============================================================

qc_checks = {

    "Panel N = 108":
        len(step04_main) == 108,

    "Countries = 27":
        step04_main["iso3"].nunique() == 27,

    "Years = 4":
        step04_main["year"].nunique() == 4,

    "Correct years":
        set(step04_main["year"]) ==
        {2021, 2023, 2024, 2025},

    "No duplicate country-year":
        step04_main.duplicated(
            ["iso3", "year"]
        ).sum() == 0,

    "VHCN missing = 0":
        step04_main["vhcn_pct"].isna().sum() == 0,

    "GDP missing = 0":
        step04_main["gdp_pc_pps"].isna().sum() == 0,

    "Log GDP missing = 0":
        step04_main["ln_gdp_pc_pps"].isna().sum() == 0
}

qc = pd.DataFrame({
    "check": qc_checks.keys(),
    "passed": qc_checks.values()
})

display(qc)

qc.to_csv(
    LOG_DIR / "Step_04_QC.csv",
    index=False,
    encoding="utf-8-sig"
)

,check,passed
0,Panel N = 108,True
1,Countries = 27,True
2,Years = 4,True
3,Correct years,True
4,No duplicate country-year,True
5,VHCN missing = 0,True
6,GDP missing = 0,True
7,Log GDP missing = 0,True


In [ ]:
# ============================================================
# DOWNLOAD STEP 04 OUTPUTS AS ZIP
# ============================================================

from pathlib import Path
import zipfile
try:
    from google.colab import files
except ImportError:
    files = None

step04_files = [
    PROJECT_ROOT / "03_processed_data" / "Step_04_Complementary_Enablers_EU27.csv",
    PROJECT_ROOT / "02_intermediate_data" / "Step_04_Cloud_auxiliary_EU27.csv",
    PROJECT_ROOT / "08_logs" / "Step_04_QC.csv"
]

zip_path = Path("/content/Step_04_outputs.zip")

print("Files to include:\n")

for f in step04_files:
    print(f.name, "->", f.exists())

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for f in step04_files:
        if not f.exists():
            raise FileNotFoundError(f"Missing required Step 04 output: {f}")
        z.write(f, arcname=f.name)

print("\nZIP created:")
print(zip_path)

if files is not None:
    files.download(str(zip_path))
else:
    print("Download manually:", zip_path)

Files to include:

Step_04_Complementary_Enablers_EU27.csv -> True
Step_04_Cloud_auxiliary_EU27.csv -> True
Step_04_QC.csv -> True

ZIP created:
/content/Step_04_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>